# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the FAIRˆ² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a [Croissant schema JSON-LD URL](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json):

`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset (Croissant package)
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

# Output summary info
print("Dataset Name:", metadata['name'])
print("Description:", metadata['description'])
print("Authors (@id):", [author['@id'] for author in metadata.get('author', [])])
print("Citations (@id):", [citation['@id'] for citation in metadata.get('citation', [])])
print("Distribution file objects (@id):", [dist['@id'] for dist in metadata.get('distribution', [])])
print("Keywords:", metadata['keywords'])

## 2. Data Overview
Explore available record sets, their fields, and the IDs for referencing within the dataset schema.

Below, we enumerate the dataset's record sets, fields, and columns by their `@id` to facilitate referencing in subsequent steps.

Note: `mlcroissant` exposes these via the `dataset.record_sets` structure.

In [ ]:
# Get available record sets and enumerate their ids and fields
record_sets = list(dataset.record_sets.keys())
print("Available RecordSet @id values:")
for record_set_id in record_sets:
    print(f"  - {record_set_id}")
    fields = dataset.record_sets[record_set_id].fields
    print("    Fields (@id):")
    for field_id, field_obj in fields.items():
        print(f"      - {field_id}: {field_obj.data_type}")
        columns = field_obj.columns
        if columns:
            print("        Columns (@id):")
            for col in columns:
                print(f"          - {col['@id']}")
    print()

## 3. Data Extraction
We load data from each available record set into a Pandas DataFrame using their `@id` values. This allows structured analysis, referencing fields and columns via `@id` throughout.

Below, we loop through all record sets, printing the first few rows and columns for one as an example.

In [ ]:
# Extract data from all record sets
dataframes = {}

for record_set_id in record_sets:
    # Load all records for this record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"RecordSet @id: {record_set_id}")
    print("Fields (@id):", df.columns.tolist())
    print("Preview:")
    print(df.head(), "\n")

# For demonstration, pick the first available record set
if record_sets:
    demo_record_set_id = record_sets[0]
    print("First record set used for further EDA:")
    print(f"@id = {demo_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records by numeric field, normalize values, categorize/group by another field, and show examples.

For demonstration, we select a numeric field and group field from the first record set.

In [ ]:
# Choose record set and fields for EDA
record_set_id = demo_record_set_id
df = dataframes[record_set_id]

print("Columns available in selected RecordSet:", df.columns.tolist())

# Attempt to locate numeric and groupable fields
numeric_field = None
group_field = None
for field_id, field_obj in dataset.record_sets[record_set_id].fields.items():
    if field_obj.data_type in ['Float', 'Integer', 'Number']:
        numeric_field = field_id
    if field_obj.data_type in ['Text', 'Boolean'] and group_field is None:
        group_field = field_id

print(f"Numeric field (@id): {numeric_field}")
print(f"Group field (@id): {group_field}")

if numeric_field:
    # Filter records by arbitrary threshold, normalize
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping
    if group_field and group_field in df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped (mean) by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize numeric field distributions and relationships between fields in the selected record set. For demonstration, we plot a histogram and a boxplot of the numeric field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Check for numeric field and plot
if numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=10, kde=True)
    plt.title(f'Histogram of {numeric_field} (@id)')
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    plt.figure(figsize=(7, 5))
    sns.boxplot(y=df[numeric_field].dropna())
    plt.title(f'Boxplot of {numeric_field} (@id)')
    plt.ylabel(numeric_field)
    plt.show()

    if group_field and group_field in df.columns:
        plt.figure(figsize=(10, 6))
        sns.barplot(x=group_field, y=numeric_field, data=df)
        plt.title(f'{numeric_field} by {group_field} (@id)')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()

## 6. Conclusion
Through this notebook, we explored the FAIRˆ² dataset package loaded via Croissant schema using `mlcroissant`.

- **Record Sets, Fields, and Columns** were referenced via their `@id`.
- Data extraction, EDA, and visualization used these identifiers for reproducible, schema-compliant workflows.
- Further research may extend analyses with domain-specific hypotheses and tailored processing.

For more information, review the Croissant schema at [https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json](https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json) and see [`mlcroissant`](https://github.com/mlcommons/croissant) documentation.